# ЛР3: SUPIR с маской в семплировании

Минимальный SR-inpainting на SUPIR-v0Q. Маска задаётся в разрешении **входного LQ**: 1 — улучшать, 0 — сохранить bicubic-увеличение. На каждом шаге EDM предсказание вне маски заменяется латентом bicubic-изображения. После декодирования выполняется точная пиксельная сборка, так как VAE и Wavelet могут менять область вне маски. Нужны Linux, CUDA и веса; полный инференс здесь заранее не выполнялся.


In [ ]:
from pathlib import Path
import os
import sys
import tempfile

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
assert (ROOT / 'SUPIR').is_dir(), 'Запустите ноутбук из корня репозитория или notebooks/'
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)

import torch
import torch.nn.functional as F
import numpy as np
from io import BytesIO
from PIL import ImageFilter
from PIL import Image
from omegaconf import OmegaConf
from pytorch_lightning import seed_everything
import CKPT_PTH

CKPT_PTH.SDXL_CLIP1_PATH = None  # локальный путь, если нет доступа к Hugging Face
CKPT_PTH.SDXL_CLIP2_CKPT_PTH = None
from SUPIR.util import create_SUPIR_model, PIL2Tensor, Tensor2PIL, convert_dtype
from sgm.util import instantiate_from_config
from SUPIR.utils.colorfix import wavelet_reconstruction

assert torch.cuda.is_available(), 'Эта версия SUPIR требует CUDA'
DEVICE = 'cuda:0'

## Модель и параметры

Поместите SDXL и SUPIR-v0Q в `checkpoints/`. Для эксперимента положите минимум три HR-изображения размером не менее 1024×1024 в `examples/gt/`. Из каждого строятся LQ 4× bicubic и LQ с размытием/JPEG. Запуск требует CUDA; дополнительные зависимости для метрик указаны в README.


In [ ]:
SDXL_CKPT = ROOT / 'checkpoints/sd_xl_base_1.0_0.9vae.safetensors'
SUPIR_CKPT = ROOT / 'checkpoints/SUPIR-v0Q.ckpt'
INPUT_IMAGE = ROOT / 'examples/input.png'
GT_DIR = ROOT / 'examples/gt'
UPSCALE = 4
MIN_SIZE = 1024
MAX_IMAGES = 3
STEPS = 50
SEED = 1234
RESTORATION_SCALE = -1.0
CFG_SCALE = 4.0
CFG_SCALE_START = 1.0
CONTROL_SCALE = 1.0
S_CHURN = 5.0
S_NOISE = 1.01
PROMPT = ''  # описание содержимого вручную; качественные слова есть в конфигурации

for path in (SDXL_CKPT, SUPIR_CKPT):
    if not path.is_file():
        raise FileNotFoundError(path)

config = OmegaConf.load(ROOT / 'options/SUPIR_v0.yaml')
config.SDXL_CKPT = str(SDXL_CKPT)
config.SUPIR_CKPT = None
config.SUPIR_CKPT_Q = str(SUPIR_CKPT)
with tempfile.TemporaryDirectory() as tmpdir:
    config_path = Path(tmpdir) / 'supir_notebook.yaml'
    OmegaConf.save(config, config_path)
    model = create_SUPIR_model(str(config_path), SUPIR_sign='Q')

model.ae_dtype = convert_dtype('bf16')
model.model.dtype = convert_dtype('fp16')
model = model.to(DEVICE).eval()

## Маска и шаг EDM

При маске `m` предсказание чистого латента становится `m * prediction + (1-m) * z_ref`, где `z_ref` — латент bicubic-увеличенного LQ. Маска переносится из разрешения входного изображения в разрешение латента усреднением. Нулевая маска сразу возвращает bicubic-изображение; единичная даёт обычный SUPIR. На выходе `Image.composite` строго сохраняет пиксели вне маски.


In [ ]:
def make_masks(image):
    w, h = image.size
    mixed = np.zeros((h, w), dtype=np.uint8)
    mixed[h // 4:3 * h // 4, w // 4:3 * w // 4] = 1
    return {
        'zero': np.zeros((h, w), dtype=np.uint8),
        'one': np.ones((h, w), dtype=np.uint8),
        'mixed': mixed,
    }


@torch.inference_mode()
def sample_latent(model, lq, mask, prompt=''):
    seed_everything(SEED)
    z_control = model.encode_first_stage_with_denoise(lq, use_sample=False)
    stage1 = model.decode_first_stage(z_control)
    center = model.encode_first_stage(stage1)
    z_ref = model.encode_first_stage(lq)
    mask_z = F.interpolate(mask.float(), size=z_control.shape[-2:], mode='area').to(z_control.dtype)
    cond, uncond = model.prepare_condition(z_control, [prompt], model.p_p, model.n_p, len(lq))

    sampler_params = model.sampler_config.params
    discretization = instantiate_from_config(sampler_params.discretization_config)
    guider_config = OmegaConf.create(OmegaConf.to_container(sampler_params.guider_config, resolve=True))
    guider_config.params.scale = CFG_SCALE_START
    guider_config.params.scale_min = CFG_SCALE
    guider = instantiate_from_config(guider_config)
    sigmas = discretization(STEPS, device=lq.device)

    x = torch.randn_like(z_control) * torch.sqrt(1 + sigmas[0] ** 2)
    sigma_max = 14.6146
    for step in range(len(sigmas) - 1):
        sigma = sigmas[step].expand(len(x))
        next_sigma = sigmas[step + 1].expand(len(x))
        gamma = min(S_CHURN / (len(sigmas) - 1), 2 ** 0.5 - 1)
        sigma_hat = sigma * (1 + gamma)
        if gamma > 0:
            x = x + torch.randn_like(x) * S_NOISE * (sigma_hat**2 - sigma**2).sqrt()[:, None, None, None]

        model_inputs = guider.prepare_inputs(x, sigma_hat, cond, uncond)
        raw = model.denoiser(model.model, *model_inputs, CONTROL_SCALE)
        prediction = guider(raw, sigma_hat)
        if RESTORATION_SCALE > 0 and next_sigma[0] > 0.05:
            weight = (sigma / sigma_max) ** RESTORATION_SCALE
            prediction = prediction - (prediction - center) * weight[:, None, None, None]

        prediction = mask_z * prediction + (1 - mask_z) * z_ref
        derivative = (x - prediction) / sigma_hat[:, None, None, None]
        x = x + (next_sigma - sigma_hat)[:, None, None, None] * derivative

    restored = model.decode_first_stage(x)
    return wavelet_reconstruction(restored, stage1)


@torch.inference_mode()
def restore_image(model, image, mask, prompt=''):
    image = image.convert('RGB')
    mask = np.asarray(mask)
    if mask.shape != (image.height, image.width):
        raise ValueError('Маска должна иметь форму (H, W) входного LQ')
    values = set(np.unique(mask).tolist())
    if not values <= {0, 1}:
        raise ValueError('Маска должна быть бинарным массивом из 0 и 1')

    lq, out_h, out_w = PIL2Tensor(image, upsacle=UPSCALE, min_size=MIN_SIZE)
    reference = image.resize((out_w, out_h), Image.Resampling.BICUBIC)
    if values == {0}:
        return reference
    mask_tensor = torch.from_numpy(np.array(mask, copy=True)).float()[None, None].to(DEVICE)
    restored = sample_latent(model, lq.unsqueeze(0).to(DEVICE), mask_tensor, prompt)
    supir = Tensor2PIL(restored[0], out_h, out_w)
    output_mask = Image.fromarray((mask * 255).astype(np.uint8)).resize(
        reference.size, Image.Resampling.NEAREST)
    return Image.composite(supir, reference, output_mask)


## Одно изображение

Поместите LQ-файл в `examples/input.png`. Выберите `MASK_KIND` из `zero`, `one`, `mixed`; результат сохранится в `results/`.


In [ ]:
MASK_KIND = 'mixed'
image = Image.open(INPUT_IMAGE).convert('RGB')
mask = make_masks(image)[MASK_KIND]
result = restore_image(model, image, mask, PROMPT)
output_path = ROOT / f'results/{INPUT_IMAGE.stem}_{MASK_KIND}.png'
output_path.parent.mkdir(parents=True, exist_ok=True)
result.save(output_path)
display(Image.fromarray(mask * 255), result)


## Эксперимент: три маски × две деградации

Берём три HR-изображения из `examples/gt/`, центрируем кроп 1024×1024 и получаем LQ 256×256. Сохраняются GT, LQ, bicubic, маски и результаты. У разных условий одинаковые параметры и seed. Эта ячейка запускает тяжёлый инференс.


In [ ]:
def make_lq(hr, degradation):
    small = (hr.width // UPSCALE, hr.height // UPSCALE)
    if degradation == 'bicubic':
        return hr.resize(small, Image.Resampling.BICUBIC)
    if degradation == 'blur_jpeg':
        blurred = hr.filter(ImageFilter.GaussianBlur(1.4))
        lq = blurred.resize(small, Image.Resampling.BICUBIC)
        stream = BytesIO()
        lq.save(stream, format='JPEG', quality=45)
        stream.seek(0)
        return Image.open(stream).convert('RGB')
    raise ValueError(degradation)


paths = sorted(p for p in GT_DIR.iterdir() if p.suffix.lower() in {'.png', '.jpg', '.jpeg'})
if len(paths) < MAX_IMAGES:
    raise ValueError(f'Нужно минимум {MAX_IMAGES} HR-изображения в {GT_DIR}')

experiment_dir = ROOT / 'results/lab3'
experiment_dir.mkdir(parents=True, exist_ok=True)
cases = []
for path in paths[:MAX_IMAGES]:
    hr = Image.open(path).convert('RGB')
    if min(hr.size) < MIN_SIZE:
        raise ValueError(f'HR-изображение слишком маленькое: {path}')
    left = (hr.width - MIN_SIZE) // 2
    top = (hr.height - MIN_SIZE) // 2
    hr = hr.crop((left, top, left + MIN_SIZE, top + MIN_SIZE))
    for degradation in ('bicubic', 'blur_jpeg'):
        name = f'{path.stem}_{degradation}'
        lq = make_lq(hr, degradation)
        baseline = lq.resize(hr.size, Image.Resampling.BICUBIC)
        hr.save(experiment_dir / f'{name}_gt.png')
        lq.save(experiment_dir / f'{name}_lq.png')
        baseline.save(experiment_dir / f'{name}_bicubic.png')
        for mask_name, mask in make_masks(lq).items():
            output = restore_image(model, lq, mask, PROMPT)
            Image.fromarray(mask * 255).save(experiment_dir / f'{name}_{mask_name}_mask.png')
            output.save(experiment_dir / f'{name}_{mask_name}.png')
            cases.append((name, mask_name))
print(f'Сохранено {len(cases)} результатов в {experiment_dir}')


## Метрики

PSNR и LPIPS сравнивают результат с HR; CLIP-IQA оценивает изображение без эталона. Для смешанной маски метрики считаются по всему кадру; дополнительно сохраняется максимальная разность вне маски относительно bicubic (должна быть 0). LPIPS и CLIP-IQA работают на CPU, чтобы не занимать память SUPIR на GPU. Первая инициализация метрик может загрузить дополнительные веса.


In [ ]:
import csv
from torchmetrics.image.lpip import LearnedPerceptualImagePatchSimilarity
from torchmetrics.multimodal.clip_iqa import CLIPImageQualityAssessment

lpips_metric = LearnedPerceptualImagePatchSimilarity(net_type='alex', normalize=True).eval()
clip_iqa_metric = CLIPImageQualityAssessment().eval()

def tensor01(image):
    array = np.asarray(image.convert('RGB'), dtype=np.float32) / 255.0
    return torch.from_numpy(array.copy()).permute(2, 0, 1).unsqueeze(0)

rows = []
with torch.inference_mode():
    for name, mask_name in cases:
        gt = np.asarray(Image.open(experiment_dir / f'{name}_gt.png').convert('RGB'), dtype=np.float32)
        output = np.asarray(Image.open(experiment_dir / f'{name}_{mask_name}.png').convert('RGB'), dtype=np.float32)
        baseline = np.asarray(Image.open(experiment_dir / f'{name}_bicubic.png').convert('RGB'), dtype=np.float32)
        mask = np.asarray(Image.open(experiment_dir / f'{name}_{mask_name}_mask.png').resize(
            (gt.shape[1], gt.shape[0]), Image.Resampling.NEAREST))
        mse = np.mean((output - gt) ** 2)
        outside = np.abs(output - baseline)[mask == 0]
        lpips_metric.reset()
        clip_iqa_metric.reset()
        rows.append({
            'case': name, 'mask': mask_name,
            'psnr_db': float('inf') if mse == 0 else float(10 * np.log10(255 ** 2 / mse)),
            'lpips': float(lpips_metric(tensor01(Image.fromarray(output.astype(np.uint8))),
                                        tensor01(Image.fromarray(gt.astype(np.uint8)))).item()),
            'clip_iqa': float(clip_iqa_metric(tensor01(Image.fromarray(output.astype(np.uint8)))).item()),
            'max_diff_outside': float(outside.max()) if outside.size else None,
        })

metrics_path = experiment_dir / 'metrics.csv'
with metrics_path.open('w', newline='') as file:
    writer = csv.DictWriter(file, fieldnames=rows[0].keys())
    writer.writeheader()
    writer.writerows(rows)
for row in rows:
    print(row)
print(metrics_path)
